In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [5]:
import numpy as np
from pydantic import BaseModel

class layer(BaseModel):
    neurons_no: int
    activation_fn: str
    layer_bias: float

def sigmoid(z: np.ndarray) -> np.ndarray:
    """
    Sigmoid is a mathematical function that maps any real-valued number into a value between 0 and 1.
    Its characteristic "S"-shaped curve makes it particularly useful in scenarios where we need to convert outputs into probabilities.
    This function is often called the logistic function.
    """
    base = 1 + np.exp(-z)
    return 1/base

def linear(z: np.ndarray)->np.ndarray:
    """
    The linear activation function which returns the output of the weighted sum itself.
    This is to be used for regression type problem statements
    """
    return z

def step(z: np.ndarray)->np.ndarray:
    """
    threshold-based activation function that outputs a binary value based on whether the input exceeds a specific limit.
    """
    return np.where(z >= 0, 1, 0)

def relu(z: np.ndarray) -> np.ndarray:
    """
    outputs the input directly if it is positive and returns zero otherwise
    """
    return np.maximum(0, z)

activation_records = {
    'sigmoid': sigmoid,
    'linear': linear,
    'step': step,
    'relu': relu
}

def weightedSum(W: np.ndarray, X: np.ndarray, b: float):
    if W.shape!= X.shape:
        raise ValueError(
            f"Invalid input received.\nThe Expected input dimension is {len(W)}"
        )
    return np.dot(W, X) + b


class Perceptron:
    def __init__(self, dim: tuple, b: float, activation: str = 'linear'):
        if activation not in activation_records:
            available = ", ".join(activation_records.keys())

            raise ValueError(
                f"Unknown activation '{activation}'. "
                f"Available activations: {available}"
            )
        self.activate = activation_records[activation]
        self.activation_name = activation
        self.weights = np.random.randn(*dim)*0.01
        self.bias = b

    def __repr__(self):
        w_str = np.array2string(self.weights, precision=4, separator=',')
        return f"Perceptron(weights={w_str}, bias={self.bias}, activation='{self.activation_name}')"

    def get_output(self, X: np.ndarray):
        z = weightedSum(self.weights, X, self.bias)
        return self.activate(z)

    def train(self, X_train: np.ndarray, y_train: np.ndarray, epochs: int = 1000, lr: float = 0.01):
        for _ in range(epochs):
            for x, y in zip(X_train, y_train):
                delta = self.get_output(x) - y

                self.weights -= lr*delta*x
                self.bias -= lr*delta
        pass

In [7]:
df = pd.read_csv('/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [8]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [10]:
scalar = StandardScaler()

In [11]:
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

In [12]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)
print(y_train.shape)
print(y_test.shape)

(614, 8)
(154, 8)
(614,)
(154,)


In [15]:
model = Perceptron(dim=(8,), b=0.0, activation='sigmoid')

In [16]:
model.train(X_train_scaled, y_train, epochs=1000, lr=0.01)

y_pred = np.array([model.get_output(x) for x in X_test_scaled])
y_pred = (y_pred >= 0.5).astype(int)

In [17]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))

[[89  6]
 [26 33]]
              precision    recall  f1-score   support

           0       0.77      0.94      0.85        95
           1       0.85      0.56      0.67        59

    accuracy                           0.79       154
   macro avg       0.81      0.75      0.76       154
weighted avg       0.80      0.79      0.78       154

0.7922077922077922
